# TV-L1 parameter sweep

Fill in the **Settings** cell, then *Run All*. Every combination of the lists in `SWEEP` is run with `run_sweep`, which writes one folder per run under `OUTDIR`:

```
OUTDIR/
  gamma1_lambda0.05/  flow.h5  flow_quiver.mp4
  ...
  summary.csv
```

Re-running skips finished runs and resumes interrupted ones, so you can add values to a list and run again. Changing anything else (gap, frame range, `BASE`, …) needs a new `OUTDIR`. See `TVL1_Optical_Flow_Guide.pdf` for what each setting does.

In [ ]:
from pathlib import Path
from itertools import product
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import cv2
from IPython.display import Image, Video, display

from auroral_processing.flow.params import TVL1Params
from auroral_processing.flow.compute_flow import make_pairs
from auroral_processing.flow.sweep import run_sweep, run_name
from auroral_processing.flow.io import open_flow, read_config
from auroral_processing.flow.score import score_sweep
from auroral_processing.flow.benchmark import make_benchmark
from auroral_processing.products.flow_video import draw_quiver, colour_lut

# Categorical colours for runs, in fixed order (max 8 runs per line chart)
RUN_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, MUTED, GRID = "#1d2330", "#6b7280", "#e5e7eb"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED,
    "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "lines.linewidth": 2, "figure.dpi": 110,
})

## Settings

In [ ]:
# ---- Source data ----
# For the synthetic benchmark (see the end of the notebook) use SRC = BENCH, FRAMES = "synth",
# TIMES = "synth_t", GAP = 1, STRIDE = 2 and a new OUTDIR.
SRC = Path.cwd().parent / "data" / "2013-03-27-CamSer7196_015400-015600-copy.h5"
FRAMES = "v4"                  # frame dataset (N, H, W)
TIMES = "v4_ut1_unix"          # time dataset (N,)

# ---- Frame pairs ----
GAP = 4                        # i1 = i0 + GAP
STRIDE = None                  # spacing between pairs (None = GAP)
START, STOP = 0, None          # frame range; e.g. STOP = 40 for a quick trial

# ---- Sweep: every combination of these lists is run ----
# Names: lambda, gamma, tau, theta, scales, scale_step, warps, epsilon,
#        inner_iterations, outer_iterations, median_filtering
SWEEP = {
    "gamma": [1, 3],
    "lambda": [0.05, 0.15, 0.4],
}
BASE = {"scale_step": 0.5}     # fixed TV-L1 settings for every run (0.5 tracks fast lanes; see the guide)

# ---- Run options ----
INPUT_DTYPE = "float32"        # "uint8" reproduces the old prototype's input
CLIP = (0.5, 99.5)             # percentiles for the global input scaling
METRICS = ["warp_mae", "zero_mae"]   # add "fb_error" (doubles run time)
WARM_START = False
WORKERS = 3                    # 3-4 is best on a 12-core machine

# ---- Videos (rendered per run; [] for none) ----
VIDEOS = ["quiver"]            # "quiver" and/or "speed"
VIDEO_KW = dict(grid=32, fps=10, vmax=25)   # fixed vmax (px/pair) so videos are comparable; None = per-video

OUTDIR = Path("sweeps") / f"{FRAMES}_gap{GAP}"


## Preview
Check the runs and pair count before starting. Nothing is computed here.

In [ ]:
base_params = TVL1Params().replace(**BASE)
combos = [dict(zip(SWEEP, values)) for values in product(*SWEEP.values())]

with h5py.File(SRC, "r") as f:
    n_frames, H, W = f[FRAMES].shape
pairs = make_pairs(n_frames, GAP, STRIDE, START, STOP)

preview = pd.DataFrame([{"run": run_name(c), **c,
                         "status": "done" if (OUTDIR / run_name(c) / "flow.h5").exists() else "to run"}
                        for c in combos])
print(f"{SRC.name}: {FRAMES} {n_frames} frames of {H}x{W}")
print(f"{len(pairs)} pairs per run (frames {pairs[0, 0]}->{pairs[0, 1]} ... {pairs[-1, 0]}->{pairs[-1, 1]}), "
      f"{len(combos)} runs, {len(pairs) * len(combos)} pairs in total")
print(f"output: {OUTDIR.resolve()}")
preview

## Run the sweep
Progress bars show each run. Interrupting the kernel is safe; run this cell again to resume.

In [ ]:
summary_csv = run_sweep(
    SRC, SWEEP, OUTDIR,
    base_params=base_params,
    videos=VIDEOS,
    video_kw=VIDEO_KW,
    frames=FRAMES, times=TIMES,
    gap=GAP, stride=STRIDE, start=START, stop=STOP,
    clip_percentiles=CLIP, input_dtype=INPUT_DTYPE,
    metrics=METRICS, warm_start=WARM_START, workers=WORKERS,
)
summary_csv

## Results
`warp_mae`: mean brightness error after warping frame i1 back onto i0 with the flow (lower = flow explains the change better). `zero_mae`: the same with no motion, a baseline. `fb_error`: forward-backward disagreement in px (only if in `METRICS`).

> With γ > 0 TV-L1 may explain brightness change as illumination rather than motion, which `warp_mae` doesn't credit. Compare `warp_mae` within the same γ; use `fb_error` and the videos across γ.

In [ ]:
df = pd.read_csv(summary_csv)
swept = [c for c in SWEEP]
metric_cols = [c for c in df.columns if c.endswith("_mean") or c.endswith("_median")]
df[["run", *swept, "pairs_done", "complete", *metric_cols]].sort_values("warp_mae_mean").style.format(precision=4)

In [ ]:
# Mean warp residual per run, best at the top; dashed line = no-motion baseline
ranked = df.sort_values("warp_mae_mean", ascending=False)
fig, ax = plt.subplots(figsize=(7, 0.38 * len(ranked) + 1.2))
ax.barh(ranked["run"], ranked["warp_mae_mean"], color=RUN_COLOURS[0], height=0.6)
baseline = df["zero_mae_mean"].mean()
ax.axvline(baseline, color=MUTED, linestyle="--", linewidth=1.2)
ax.text(baseline, len(ranked) - 0.4, " no-motion baseline", color=MUTED, va="bottom", fontsize=9)
for y, v in enumerate(ranked["warp_mae_mean"]):
    ax.text(v, y, f" {v:.4f}", va="center", color=INK, fontsize=8.5)
ax.set_xlabel("mean warp_mae (scaled brightness, lower is better)")
ax.set_xlim(0, max(baseline, ranked["warp_mae_mean"].max()) * 1.18)
ax.grid(axis="y", visible=False)
ax.set_title("Warp residual by run", loc="left", color=INK)
plt.tight_layout(); plt.show()

In [ ]:
# If exactly two settings are swept: grid of mean warp residual
if len(swept) == 2:
    a, b = swept
    grid = df.pivot_table(index=a, columns=b, values="warp_mae_mean")
    fig, ax = plt.subplots(figsize=(1.3 * grid.shape[1] + 2.4, 0.9 * grid.shape[0] + 1.4))
    im = ax.imshow(grid.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(grid.shape[1]), [f"{v:g}" for v in grid.columns]); ax.set_xlabel(b)
    ax.set_yticks(range(grid.shape[0]), [f"{v:g}" for v in grid.index]); ax.set_ylabel(a)
    ax.grid(False)
    for (i, j), v in np.ndenumerate(grid.values):
        dark = v > np.nanmean(grid.values)
        ax.text(j, i, f"{v:.4f}", ha="center", va="center", fontsize=9, color="white" if dark else INK)
    fig.colorbar(im, ax=ax, label="mean warp_mae")
    ax.set_title(f"Mean warp residual: {a} x {b}", loc="left", color=INK)
    plt.tight_layout(); plt.show()
else:
    print(f"{len(swept)} swept settings; the grid view needs exactly 2.")

### Per-pair metrics over time
Pick up to 8 runs to compare (default: the first 8 in sweep order).

In [ ]:
COMPARE = list(df["run"])[:8]     # e.g. ["gamma1_lambda0.15", "gamma3_lambda0.15"]
METRIC = "warp_mae"                # or "fb_error" if computed

fig, ax = plt.subplots(figsize=(9, 4))
for colour, run in zip(RUN_COLOURS, COMPARE):
    with open_flow(OUTDIR / run / "flow.h5") as f:
        ax.plot(f["i0"][:], f[f"metrics/{METRIC}"][:], color=colour, label=run)
        zero = f["metrics/zero_mae"][:] if METRIC == "warp_mae" and "metrics/zero_mae" in f else None
        i0 = f["i0"][:]
if zero is not None:
    ax.plot(i0, zero, color=MUTED, linestyle="--", linewidth=1.2, label="no-motion baseline")
ax.set_xlabel("first frame of pair (i0)"); ax.set_ylabel(METRIC)
ax.legend(frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"{METRIC} per pair", loc="left", color=INK)
plt.tight_layout(); plt.show()

## Compare one pair across runs
The same pair from every run, with one shared arrow scale and colour scale (`VMAX`), so differences are due to the settings alone.

In [ ]:
PAIR = None                   # pair index; None = middle of the run
VMAX = VIDEO_KW.get("vmax") or 15   # px/pair at the top of the colour scale
GRID_ARROWS = VIDEO_KW.get("grid", 32)
COLS = 3

runs = list(df["run"])
lut = colour_lut("turbo")
rows = -(-len(runs) // COLS)
fig, axes = plt.subplots(rows, COLS, figsize=(4.2 * COLS, 4.5 * rows), squeeze=False, layout="constrained")
with h5py.File(SRC, "r") as src:
    for ax, run in zip(axes.flat, runs):
        with open_flow(OUTDIR / run / "flow.h5") as f:
            k = len(f["i0"]) // 2 if PAIR is None else PAIR
            i0, i1 = int(f["i0"][k]), int(f["i1"][k])
            lo, hi = f.attrs["lo"], f.attrs["hi"]
            bg = (np.clip((src[FRAMES][i0].astype(np.float32) - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
            step = max(max(bg.shape) // GRID_ARROWS, 1)
            img = draw_quiver(bg, f["flow"][k], step, 0.9 * step / VMAX, VMAX, lut)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(run, fontsize=10, color=INK); ax.axis("off")
for ax in axes.flat[len(runs):]:
    ax.axis("off")
fig.suptitle(f"Pair {k}: frames {i0} -> {i1}   (arrow scale fixed, colour 0-{VMAX:g} px/pair)", x=0.01, ha="left", color=INK)
plt.show()

## Watch a run's video

In [ ]:
RUN = df.sort_values("warp_mae_mean")["run"].iloc[0]   # best warp residual; or type a run name
KIND = VIDEOS[0] if VIDEOS else None

video = OUTDIR / RUN / f"flow_{KIND}.mp4"
print(video)
display(Video(video, embed=True, width=560)) if KIND and video.exists() else print("no video for this run")

## Score the runs
`warp_mae` keeps falling as lambda rises (the flow fits noise), so it can't choose lambda on its own. `score_sweep` splits each frame into **structure** (strong gradients: lanes and edges), **stars** and **diffuse glow**, and scores each run from its saved flow (nothing is recomputed):

- `warp_diffuse` below `noise_floor`: the flow is fitting noise
- `speed_diffuse_p50`: motion where there is little to track (mostly spurious)
- `temporal_change_lane`, `rough_diffuse`: jitter in time and space
- `star_over_ring` < 1: static stars pin the flow around them

If `SRC` is a synthetic benchmark it also scores against the known flow (`epe_lane`, `speed_recovery`, `structure_corr`, `false_bright`, `combined_rank`, ...). Pass `source=` if the sweep ran on another computer.

In [ ]:
scores = pd.read_csv(score_sweep(OUTDIR, source=SRC, progress=False))
display(Image(OUTDIR / "scores_heatmaps.png"))
sort_by = "combined_rank" if "combined_rank" in scores else "speed_diffuse_p50"
scores.sort_values(sort_by).round(4)

## Synthetic benchmark (known flow)
Real frames whose dark lanes are moved by a **known** flow (rigid at several speeds, shear, vortex, and optionally a smoothed real flow), with stars kept still, still patches that only change brightness, and noise matched to the source. Build it once, then set `SRC = BENCH`, `FRAMES = "synth"`, `TIMES = "synth_t"`, `GAP = 1`, `STRIDE = 2` and a new `OUTDIR` in the Settings cell and run the sweep and scoring cells again.

In [ ]:
BENCH = Path("bench_v4.h5")
REALISTIC = None   # e.g. Path("sweeps/v4_gap4/gamma1_lambda0.1/flow.h5") for an extra realistic case

if not BENCH.exists():
    make_benchmark(SRC, BENCH, frames=FRAMES, speeds=(5, 15, 25), realistic_flow=REALISTIC)
with h5py.File(BENCH) as f:
    print(len(f["case"]), "pairs:", sorted(set(f["case"].asstr()[:])), "from frames", sorted(set(f["base"][:])))

## Load a run's flow for your own analysis
`flow[k]` is (H, W, 2) in px per pair: `[..., 0]` = u (+x right), `[..., 1]` = v (+y **down**). Divide by `t1 - t0` for px/s. Read one pair at a time; the whole dataset can be GBs.

In [ ]:
with open_flow(OUTDIR / RUN / "flow.h5") as f:
    config = read_config(f)
    k = len(f["i0"]) // 2
    u, v = f["flow"][k, ..., 0], f["flow"][k, ..., 1]
    dt = f["t1"][k] - f["t0"][k]

print(config["tvl1"])
print(f"pair {k}: median speed {np.median(np.hypot(u, v)):.2f} px/pair = {np.median(np.hypot(u, v)) / dt:.2f} px/s")